# Vòng 2 — model mạnh hơn cho luồng mô tả tự do (Colab GPU)

**Vì sao:** model v1 chỉ học một kiểu truy vấn (phần "thực tế") → thắng lớn trên bộ paraphrase nhưng gần như không hơn model gốc trên 50 truy vấn viết lại. Vòng 2 thêm cặp **tiêu đề → issue** của ~6.200 issue ngoài bộ đo (17.236 dòng, có bản bỏ dấu) và thử e5-large.

| thành phần | lựa chọn |
|---|---|
| dense | e5-task (v1, train lại ở đây) / e5-task2 (e5-base, v2) / e5-large-task2 (e5-large, v2) |
| reranker | ce-task (v1, train lại) / ce-task2 (v2) |
| trộn | blend@0.3 / blend@0.5 |

**Chọn trên dev của 2 bộ đo × có dấu/không dấu (4 lát):** R@10 không thấp hơn `e5-task|ce-task|0.3` có ý nghĩa ở cả 4 lát, rồi lấy MRR@10 trung bình cao nhất (`pipeline/final_eval2.py`). Test chạy một lần; cuối cùng kiểm tra 50 truy vấn viết tay.

**Trước khi chạy**
1. GPU: T4 chạy được nhưng lâu (~3–4 giờ tổng). Nếu có L4/A100 thì chọn, nhanh hơn nhiều.
2. Chuẩn bị `colab/round2_bundle.zip`.
3. **Dữ liệu sẽ rời máy sang Google Colab.**
4. Colab ngắt khi tab không hoạt động lâu — để tab mở.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1" datasets accelerate

In [ ]:
from google.colab import files
up = files.upload()            # chon round2_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## 0. Chạy thử train (vài bước mỗi model)

In [ ]:
!python pipeline/train_task_models.py --model bi --pairs data/processed/task_pairs_v2.jsonl --out /tmp/s1 --max-steps 3 --limit 100
!python pipeline/train_task_models.py --model bi --pairs data/processed/task_pairs_v2.jsonl --base intfloat/multilingual-e5-large --batch 16 --out /tmp/s2 --max-steps 3 --limit 100
!python pipeline/train_task_models.py --model ce --pairs data/processed/task_pairs_v2.jsonl --out /tmp/s3 --max-steps 3 --limit 30

## 1. Train lại v1 (mốc so sánh, cùng script + seed)

In [ ]:
!python pipeline/train_task_models.py --model bi --pairs data/processed/task_pairs.jsonl --out models/e5-base-task
!python pipeline/train_task_models.py --model ce --pairs data/processed/task_pairs.jsonl --out models/bge-reranker-base-task

## 2. Train v2

In [ ]:
!python pipeline/train_task_models.py --model bi --pairs data/processed/task_pairs_v2.jsonl --out models/e5-base-task2
!python pipeline/train_task_models.py --model ce --pairs data/processed/task_pairs_v2.jsonl --out models/bge-reranker-base-task2

In [ ]:
!python pipeline/train_task_models.py --model bi --pairs data/processed/task_pairs_v2.jsonl --base intfloat/multilingual-e5-large --batch 16 --out models/e5-large-task2
!ls models

## 3. Dev: đo lưới trên 2 bộ đo và chọn

In [ ]:
!python pipeline/final_eval2.py --split dev --device cuda
!head -3 data/processed/final2_choice.json

## 4. Test: chạy MỘT lần

In [ ]:
!python pipeline/final_eval2.py --split test --device cuda

## 5. Kiểm tra 50 truy vấn viết tay: v1 (train lại) vs lựa chọn

In [ ]:
import json
D = {"e5-task": "models/e5-base-task", "e5-task2": "models/e5-base-task2", "e5-large-task2": "models/e5-large-task2"}
R = {"ce-task": "models/bge-reranker-base-task", "ce-task2": "models/bge-reranker-base-task2"}
chosen = json.load(open("data/processed/final2_choice.json"))["chosen"]
dn, rn, _ = chosen.split("|")
print("lua chon:", chosen)
!python pipeline/eval_handwritten.py --device cuda --system v1 models/e5-base-task models/bge-reranker-base-task --system chon {D[dn]} {R[rn]} --out data/processed/handwritten_eval_round2.json | head -20

## 6. Tải kết quả + model được chọn

Ô đầu tải file kết quả (nhỏ). Ô sau nén model được chọn; thử lưu Google Drive, nếu mount lỗi thì tải trực tiếp (mỗi model ~1–2,2 GB).

In [ ]:
!zip -q round2_results.zip data/processed/final2_*.json data/processed/handwritten_eval_round2.json models/*/task_meta.json
files.download("round2_results.zip")

In [ ]:
need = sorted({D[dn], R[rn]})
for p in need:
    name = p.split("/")[-1]
    !zip -qr {name}.zip {p}
!ls -la *.zip
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    for p in need:
        name = p.split("/")[-1]
        !cp {name}.zip /content/drive/MyDrive/
    print("da luu vao MyDrive")
except Exception as e:
    print("mount Drive loi:", e, "-> tai truc tiep")
    for p in need:
        files.download(p.split("/")[-1] + ".zip")